# Ordered logit

## Model

An unobserved continuous response determines the observed ordered category:

$$
y_n^*=\beta_1x_{n1}+\beta_2x_{n2}+\varepsilon_n,\qquad
y_n=q\ \text{if}\ \tau_{q-1}<y_n^*\leq\tau_q.
$$

With logistic errors,

$$
\Pr(y_n=q)=
\Lambda(\tau_q-\boldsymbol x_n^\mathsf T\boldsymbol\beta)
-\Lambda(\tau_{q-1}-\boldsymbol x_n^\mathsf T\boldsymbol\beta).
$$

The example estimates two covariate effects and three ordered thresholds for
four outcome categories.

This notebook is self-contained and was executed in the repository's Office
validation environment. Install the released package with
`pip install torchdcm`, then select CPU or CUDA through the `device` argument.


In [1]:
import numpy as np
import torch

from IPython.display import HTML, display

from torchdcm import Beta, OrderedChoiceDataset, OrderedLogit
# Use double precision and a fixed seed for stable, reproducible estimation.
torch.set_default_dtype(torch.float64)
torch.manual_seed(7)
# Use CUDA automatically when available; set device = "cpu" to force CPU.
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"TorchDCM example running on {device}")


TorchDCM example running on cuda


In [2]:
# Simulate a latent response with logistic errors and four categories.
rng = np.random.default_rng(18)
n_obs = 800
x_1 = rng.normal(size=n_obs)
x_2 = rng.normal(size=n_obs)
u = rng.uniform(size=n_obs)
logistic_error = np.log(u) - np.log1p(-u)
latent_utility = 0.90 * x_1 - 0.60 * x_2 + logistic_error
y = np.digitize(latent_utility, bins=[-1.20, -0.20, 0.90])

# Store integer outcomes, covariates, weights, and category labels.
data = OrderedChoiceDataset(
    y=torch.as_tensor(y, dtype=torch.long),
    x={
        "x_1": torch.as_tensor(x_1),
        "x_2": torch.as_tensor(x_2),
    },
    weights=torch.ones(n_obs),
    categories=[0, 1, 2, 3],
)
print("Category counts:", dict(zip(*np.unique(y, return_counts=True))))


Category counts: {np.int64(0): np.int64(220), np.int64(1): np.int64(142), np.int64(2): np.int64(163), np.int64(3): np.int64(275)}


In [3]:
# Specify the latent index and three increasing cut points.
latent = (
    Beta("B_X1", init=0.50) * "x_1"
    + Beta("B_X2", init=-0.30) * "x_2"
)
thresholds = [
    Beta("TAU_1", init=-1.00),
    Beta("TAU_2", init=-0.10),
    Beta("TAU_3", init=0.80),
]
# Use the logistic CDF to estimate ordered-category probabilities.
model = OrderedLogit(latent, thresholds, device=device, max_iter=80)
result = model.fit(data)
# Render threshold estimates and the remaining model diagnostics.
display(HTML(result.report().to_html()))


Model family,OrderedLogit
Estimation objective,Maximum log likelihood
TorchDCM version,0.1.0
PyTorch version,2.12.1+cu130
Python version,3.12.13
Operating system,Linux-6.17.0-35-generic-x86_64-with-glibc2.39
Device,cuda
Tensor dtype,float64
Optimizer,torch.optim.LBFGS
Maximum iterations,80
Line search,strong_wolfe
